### Project By- Pranali for DASA TECHNOWORLD PVT. LTD.

# 1. Project Introduction

This notebook prepares the raw farm data for machine learning by:
- Reading multiple Excel sheets
- Cleaning and validating the data
- Aggregating farm-level metrics
- Creating derived business features
- Saving the processed dataset

# 2. Load Required Libraries

In [1]:
import pandas as pd
file_path = "../data/raw/CLOLINE_Client_Project_Data.xlsx"

farmers = pd.read_excel(file_path, sheet_name="Farmers")
production = pd.read_excel(file_path, sheet_name="Production_Batches")
feed = pd.read_excel(file_path, sheet_name="Feed_Transactions")
sales = pd.read_excel(file_path, sheet_name="Sales")
contracts = pd.read_excel(file_path, sheet_name="Contracts")

# 3. Read Raw Dataset

In [2]:
print("Farmers Shape:", farmers.shape)
print("Production Shape:", production.shape)
print("Feed Shape:", feed.shape)
print("Sales Shape:", sales.shape)
print("Contracts Shape:", contracts.shape)

Farmers Shape: (20000, 7)
Production Shape: (35000, 6)
Feed Shape: (40000, 6)
Sales Shape: (45000, 6)
Contracts Shape: (12000, 6)


In [3]:
# ============================================
# Initial Data Inspection
# ============================================

datasets = {
    "Farmers": farmers,
    "Production": production,
    "Feed": feed,
    "Sales": sales, 
    "Contracts": contracts
}

for name, df in datasets.items():
    print("="*60)
    print(f"{name} Dataset")
    print("="*60)
    print(f"Shape : {df.shape}")
    print("\nData Types:")
    print(df.dtypes)
    print("\nMissing Values:")
    print(df.isnull().sum())
    print("\nDuplicate Rows:", df.duplicated().sum())
    print("-"*60)

Farmers Dataset
Shape : (20000, 7)

Data Types:
Farmer_ID                    int64
Farmer_Name                    str
Mobile_Number                int64
State                          str
Farm_Size_Birds              int64
Experience_Years             int64
Join_Date           datetime64[us]
dtype: object

Missing Values:
Farmer_ID           0
Farmer_Name         0
Mobile_Number       0
State               0
Farm_Size_Birds     0
Experience_Years    0
Join_Date           0
dtype: int64

Duplicate Rows: 0
------------------------------------------------------------
Production Dataset
Shape : (35000, 6)

Data Types:
Batch_ID                     int64
Farmer_ID                    int64
Chick_Count                  int64
Feed_Consumed_KG             int64
Mortality_Count              int64
Production_Date     datetime64[us]
dtype: object

Missing Values:
Batch_ID            0
Farmer_ID           0
Chick_Count         0
Feed_Consumed_KG    0
Mortality_Count     0
Production_Date     0
dtype

# 4. Select Required Columns

In [4]:
farmers = farmers[
    ['Farmer_ID', 'State', 'Farm_Size_Birds', 'Experience_Years', 'Join_Date']
]

production = production[
    ['Farmer_ID', 'Chick_Count', 'Feed_Consumed_KG', 'Mortality_Count', 'Production_Date']
]

feed = feed[
    ['Farmer_ID', 'Quantity_KG', 'Feed_Cost_INR', 'Txn_Date']
]

sales = sales[
    ['Farmer_ID', 'Sale_Type', 'Quantity', 'Revenue_INR', 'Sale_Date']
]

contracts = contracts[
    ['Farmer_ID', 'Contract_Type', 'Contract_Duration_Months', 'Expected_ROI_%']
]

# 5. Aggregate Feed, Production and Sales Data

In [5]:
feed_agg = feed.groupby('Farmer_ID').agg(
    Total_Feed_Used_KG=('Quantity_KG', 'sum'),
    Total_Feed_Cost=('Feed_Cost_INR', 'sum')
).reset_index()

In [6]:
production_agg = production.groupby('Farmer_ID').agg(
    Total_Chicks=('Chick_Count', 'sum'),
    Total_Mortality=('Mortality_Count', 'sum'),
    Total_Feed_Consumed=('Feed_Consumed_KG', 'sum')
).reset_index()

In [7]:
sales_agg = sales.groupby('Farmer_ID').agg(
    Total_Sales_Qty=('Quantity', 'sum'),
    Total_Revenue=('Revenue_INR', 'sum')
).reset_index()

In [8]:
final_df = farmers \
    .merge(production_agg, on='Farmer_ID', how='left') \
    .merge(feed_agg, on='Farmer_ID', how='left') \
    .merge(sales_agg, on='Farmer_ID', how='left') \
    .merge(contracts, on='Farmer_ID', how='left')

# 6. Create Derived Features

In [9]:
final_df['Mortality_Rate'] = final_df['Total_Mortality'] / final_df['Total_Chicks']
final_df['Profit'] = final_df['Total_Revenue'] - final_df['Total_Feed_Cost']

# 7. Merge Datasets

In [10]:
print("Merged Dataset Shape:", final_df.shape)
print("Unique Farmer IDs:", final_df["Farmer_ID"].nunique())

Merged Dataset Shape: (22957, 17)
Unique Farmer IDs: 20000


# 8. Data Cleaning & Validation

In [11]:
# ============================================
# Data Cleaning & Validation
# ============================================

print("Missing Values Before Cleaning")
print(final_df.isnull().sum())

# Numeric columns -> Median
numeric_cols = final_df.select_dtypes(include='number').columns

for col in numeric_cols:
    final_df[col] = final_df[col].fillna(final_df[col].median())

# Categorical columns -> Mode
categorical_cols = final_df.select_dtypes(include=['object', 'string']).columns

for col in categorical_cols:
    final_df[col] = final_df[col].fillna(final_df[col].mode()[0])

# Remove duplicate rows
duplicates = final_df.duplicated().sum()
print(f"\nDuplicate Rows: {duplicates}")

if duplicates > 0:
    final_df.drop_duplicates(inplace=True)

print("\nMissing Values After Cleaning")
print(final_df.isnull().sum())

Missing Values Before Cleaning
Farmer_ID                       0
State                           0
Farm_Size_Birds                 0
Experience_Years                0
Join_Date                       0
Total_Chicks                 3953
Total_Mortality              3953
Total_Feed_Consumed          3953
Total_Feed_Used_KG           3016
Total_Feed_Cost              3016
Total_Sales_Qty              2433
Total_Revenue                2433
Contract_Type               10957
Contract_Duration_Months    10957
Expected_ROI_%              10957
Mortality_Rate               3953
Profit                       5091
dtype: int64

Duplicate Rows: 40

Missing Values After Cleaning
Farmer_ID                   0
State                       0
Farm_Size_Birds             0
Experience_Years            0
Join_Date                   0
Total_Chicks                0
Total_Mortality             0
Total_Feed_Consumed         0
Total_Feed_Used_KG          0
Total_Feed_Cost             0
Total_Sales_Qty            

# 9. Final Dataset Summary

In [12]:
print("Rows :", final_df.shape[0])
print("Columns :", final_df.shape[1])

Rows : 22917
Columns : 17


In [13]:
print("\nColumn Names:")
print(final_df.columns.tolist())


Column Names:
['Farmer_ID', 'State', 'Farm_Size_Birds', 'Experience_Years', 'Join_Date', 'Total_Chicks', 'Total_Mortality', 'Total_Feed_Consumed', 'Total_Feed_Used_KG', 'Total_Feed_Cost', 'Total_Sales_Qty', 'Total_Revenue', 'Contract_Type', 'Contract_Duration_Months', 'Expected_ROI_%', 'Mortality_Rate', 'Profit']


In [14]:
print("\nStatistical Summary:")
display(final_df.describe())


Statistical Summary:


,Farmer_ID,Farm_Size_Birds,Experience_Years,Join_Date,Total_Chicks,Total_Mortality,Total_Feed_Consumed,Total_Feed_Used_KG,Total_Feed_Cost,Total_Sales_Qty,Total_Revenue,Contract_Duration_Months,Expected_ROI_%,Mortality_Rate,Profit
count,22917.000000,22917.000000,22917.000000,22917,22917.000000,22917.000000,22917.000000,22917.000000,2.291700e+04,22917.000000,2.291700e+04,22917.000000,22917.000000,22917.000000,2.291700e+04
mean,109982.539076,11810.990924,11.467077,2023-06-19 01:05:39.782694,5625.210106,315.204041,16289.988611,9273.482786,2.918153e+05,6304.410656,7.596370e+05,12.012829,16.254265,0.066263,4.535417e+05
min,100000.000000,500.000000,1.000000,2020-12-17 00:00:00,502.000000,5.000000,800.000000,200.000000,8.015000e+03,100.000000,1.523100e+04,6.000000,8.000000,0.001190,-1.127058e+06
25%,104984.000000,6054.000000,6.000000,2022-03-22 00:00:00,3504.000000,191.000000,10073.000000,5381.000000,1.722760e+05,3655.000000,4.365610e+05,12.000000,16.000000,0.040946,1.791730e+05
50%,109976.000000,11331.000000,11.000000,2023-06-19 00:00:00,4888.000000,280.000000,14364.000000,8100.000000,2.558390e+05,5592.000000,6.752780e+05,12.000000,16.000000,0.055634,3.969535e+05
75%,114998.000000,16816.000000,17.000000,2024-09-23 00:00:00,7157.000000,403.000000,20677.000000,12200.000000,3.859210e+05,8327.000000,1.005943e+06,12.000000,17.000000,0.074215,6.575950e+05
max,119999.000000,30000.000000,25.000000,2025-12-17 00:00:00,29756.000000,1731.000000,97957.000000,45179.000000,1.334664e+06,34838.000000,3.709579e+06,18.000000,25.000000,0.560311,3.623273e+06
std,5776.816292,7000.485655,6.283460,NaN,3350.788786,196.589389,9902.014366,5796.629348,1.801281e+05,3903.978263,4.691577e+05,3.538188,3.772422,0.051236,4.705827e+05


In [15]:
print("\nFirst Five Rows:")
display(final_df.head())


First Five Rows:


,Farmer_ID,State,Farm_Size_Birds,Experience_Years,Join_Date,Total_Chicks,Total_Mortality,Total_Feed_Consumed,Total_Feed_Used_KG,Total_Feed_Cost,Total_Sales_Qty,Total_Revenue,Contract_Type,Contract_Duration_Months,Expected_ROI_%,Mortality_Rate,Profit
0,100000,UP,9042,8,2022-01-31,4104.0,33.0,4101.0,8100.0,255839.0,8191.0,817700.0,Revenue_Sharing,12.0,16.0,0.008041,396953.5
1,100001,Maharashtra,17015,2,2023-06-07,1178.0,101.0,12020.0,12871.0,368959.0,5592.0,675278.0,Revenue_Sharing,12.0,16.0,0.085739,396953.5
2,100002,Tamil Nadu,14906,14,2023-07-06,3770.0,400.0,24500.0,8100.0,255839.0,10752.0,1012352.0,Buy_Back,12.0,12.0,0.106101,396953.5
3,100003,Telangana,706,19,2025-10-27,1754.0,281.0,4722.0,25057.0,850025.0,5978.0,974851.0,Revenue_Sharing,12.0,16.0,0.160205,124826.0
4,100004,Maharashtra,8072,16,2024-06-15,12844.0,318.0,21365.0,3166.0,51899.0,16567.0,1760982.0,Buy_Back,18.0,10.0,0.024759,1709083.0


In [16]:
print("Merged Dataset Shape:", final_df.shape)
print(f"Unique Farmer IDs: {final_df['Farmer_ID'].nunique()}")
print(f"Total Records: {len(final_df)}")

if final_df['Farmer_ID'].nunique() == len(final_df):
    print("✓ One record per farmer.")
else:
    print("ℹ Dataset contains multiple records per farmer (expected for multiple batches/transactions).")

Merged Dataset Shape: (22917, 17)
Unique Farmer IDs: 20000
Total Records: 22917
ℹ Dataset contains multiple records per farmer (expected for multiple batches/transactions).


# 10. Save Processed Dataset

In [17]:
column_order = [
    "Farmer_ID",
    "State",
    "Farm_Size_Birds",
    "Experience_Years",
    "Join_Date",
    "Contract_Type",
    "Contract_Duration_Months",
    "Expected_ROI_%",
    "Total_Chicks",
    "Total_Mortality",
    "Total_Feed_Consumed",
    "Total_Feed_Used_KG",
    "Total_Feed_Cost",
    "Total_Sales_Qty",
    "Total_Revenue",
    "Mortality_Rate",
    "Profit"
]

final_df = final_df[column_order]

In [18]:

final_df.to_csv("../data/processed/farm_ml_clean_data.csv", index=False)

In [19]:
final_df.to_excel("../data/processed/farm_ml_clean_data.xlsx",index=False)

# Notebook Conclusion

The raw farm datasets were successfully preprocessed by:

- Reading and inspecting all source tables
- Selecting relevant columns
- Aggregating production, feed, and sales information
- Merging datasets into a unified farm-level dataset
- Creating initial business features
- Handling missing values and duplicates
- Validating the final dataset
- Saving the processed dataset for Exploratory Data Analysis (EDA)

The processed dataset is now ready for further analysis and machine learning.